[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Testing a Data Layer &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and writes the project its
worked examples wrote: `college_models.py`, `registrar.py`, and `conftest.py` with the engine
fixture and the session fixture that rolls back. Run it first. Every task writes a test file of its
own, and the last cell removes the scratch folder.


In [1]:
import logging
import os
import re
import shutil
import subprocess
import sys
from datetime import date
from importlib.metadata import version
from pathlib import Path

import sqlalchemy
from sqlalchemy import CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, create_engine, event, func, insert, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))

os.environ["NO_COLOR"] = "1"                                        # no terminal codes in what pytest prints
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"                         # no compiled copy of a file rewritten within a second

PROJECT = SCRATCH / "registrar"
PROJECT.mkdir()


def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    report = re.sub(r"0x[0-9a-f]+", "0x...", report)                    # the memory addresses of objects
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, errors_only=False):
    """Run python -m pytest in the project folder and print its report, or only its error lines and last line."""
    lines = pytest_report(*arguments).splitlines()
    if errors_only:
        lines = [line for line in lines[:-1] if line.startswith(("E ", "FAILED", "ERROR"))] + lines[-1:]
    print("\n".join(lines))


print("pytest", version("pytest"))

PROJECT_FILES = {
    "college_models.py": r'''"""The college's tables, as classes: the models Alembic compares the database with."""
from datetime import date

from sqlalchemy import CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"
''',
    "registrar.py": r'''"""The registrar's data layer: every function takes the session it works in."""
from sqlalchemy import func, select

from college_models import Enrollment, Section, Student


class SectionFull(Exception):
    """A section with no seat left."""


def find_student(session, email):
    """The student with this email, or None."""
    return session.scalars(select(Student).where(Student.email == email)).one_or_none()


def seats_left(session, section_id):
    """How many more students a section can take."""
    enrolled = session.scalar(
        select(func.count()).select_from(Enrollment)
        .where(Enrollment.section_id == section_id, Enrollment.status == "enrolled")
    )
    return session.get(Section, section_id).capacity - enrolled


def enroll(session, student_id, section_id):
    """Enroll a student in a section that has a seat left, and commit."""
    if seats_left(session, section_id) < 1:
        raise SectionFull(f"section {section_id} has no seat left")
    session.add(Enrollment(student_id=student_id, section_id=section_id))
    session.commit()


def withdraw(session, student_id, section_id):
    """Mark an enrollment withdrawn, which gives its seat back, and commit."""
    session.get(Enrollment, (student_id, section_id)).status = "withdrawn"
    session.commit()
''',
    "conftest.py": r'''from datetime import date

import pytest
from sqlalchemy import create_engine, event
from sqlalchemy.orm import Session
from sqlalchemy.pool import StaticPool

from college_models import Base, Course, Section, Student, Term


@pytest.fixture(scope="session")
def engine():
    """A database in memory with the college's tables and a little data, built once for the whole run."""
    engine = create_engine("sqlite://", poolclass=StaticPool, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False

    Base.metadata.create_all(engine)
    with Session(engine) as session:
        session.add_all([
            Student(name="Ana Reyes", email="areyes@college.edu", program="Biology", started_on=date(2024, 8, 26)),
            Student(name="Ben Okafor", email="bokafor@college.edu", program="History", started_on=date(2025, 1, 13)),
            Student(name="Chloe Martin", email="cmartin@college.edu", program="Mathematics", started_on=date(2025, 8, 25)),
            Section(course=Course(code="STA-200", title="Statistics", department="Mathematics", credits=3),
                    term=Term(name="Fall 2026", starts_on=date(2026, 8, 24)), capacity=2),
        ])
        session.commit()
    yield engine
    engine.dispose()


@pytest.fixture
def session(engine):
    """A session for one test, inside a transaction that is rolled back when the test ends."""
    with engine.connect() as connection:
        transaction = connection.begin()
        with Session(bind=connection, join_transaction_mode="create_savepoint") as session:
            yield session
        transaction.rollback()
''',
}
for name, text in PROJECT_FILES.items():
    (PROJECT / name).write_text(text)


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}
pytest 8.4.2


**1.** A section nobody has joined.


In [2]:
(PROJECT / "test_task_1.py").write_text("""
from registrar import seats_left


def test_a_section_nobody_joined_has_every_seat(session):
    left = seats_left(session, 1)
    assert left == 2
""")
run_pytest("-q", "test_task_1.py")


.                                                                        [100%]
1 passed


The engine fixture's section has two seats, and no test before this one could have left an
enrollment behind.


**2.** The same enrollment twice.


In [3]:
(PROJECT / "test_task_2.py").write_text("""
import pytest
from sqlalchemy.exc import IntegrityError

from registrar import enroll, seats_left


def test_the_same_enrollment_twice_is_refused(session):
    enroll(session, 1, 1)
    with pytest.raises(IntegrityError, match="UNIQUE constraint failed"):
        enroll(session, 1, 1)
    session.rollback()
    left = seats_left(session, 1)
    assert left == 1
""")
run_pytest("-q", "test_task_2.py")


.                                                                        [100%]
1 passed


The first `enroll` committed, which released its savepoint, so the rollback after the refusal went
back only as far as the savepoint the second one began, and the first enrollment stayed.


**3.** `seats_left`, parametrized.


In [4]:
(PROJECT / "test_task_3.py").write_text("""
import pytest

from registrar import enroll, seats_left


@pytest.mark.parametrize("enrolled, left", [(0, 2), (1, 1), (2, 0)])
def test_seats_left(session, enrolled, left):
    for student_id in range(1, enrolled + 1):
        enroll(session, student_id, 1)
    assert seats_left(session, 1) == left
""")
run_pytest("-v", "test_task_3.py")


============================= test session starts ==============================
collecting ... collected 3 items

test_task_3.py::test_seats_left[0-2] PASSED                              [ 33%]
test_task_3.py::test_seats_left[1-1] PASSED                              [ 66%]
test_task_3.py::test_seats_left[2-0] PASSED                              [100%]

============================== 3 passed ===============================


Three tests, each enrolling its own students in its own transaction.


**4.** A fixture for a second section.


In [5]:
(PROJECT / "test_task_4.py").write_text("""
import pytest

from college_models import Course, Section, Term
from registrar import seats_left


@pytest.fixture
def second_section(session):
    section = Section(course=Course(code="CSC-101", title="Programming I", department="Computer Science", credits=3),
                      term=session.get(Term, 1), capacity=1)
    session.add(section)
    session.flush()
    return section.id


def test_a_second_section_has_one_seat(session, second_section):
    assert seats_left(session, second_section) == 1
""")
run_pytest("-q", "test_task_4.py")


.                                                                        [100%]
1 passed


`flush()` sends the `INSERT`s, which gives the section its id, without committing; the course and the
section go with the test's transaction.


**5.** The order the fixtures are set up in.


In [6]:
run_pytest("-q", "--setup-show", "test_task_4.py")



SETUP    S engine
        SETUP    F session (fixtures used: engine)
        SETUP    F second_section (fixtures used: session)
        test_task_4.py::test_a_second_section_has_one_seat (fixtures used: engine, second_section, session).
        TEARDOWN F second_section
        TEARDOWN F session
TEARDOWN S engine
1 passed


`second_section` asks for `session`, so pytest sets up the engine, then the session, then the
section, and tears them down in the opposite order.


**6.** The whole folder.


In [7]:
run_pytest("-q")


......                                                                   [100%]
6 passed


Six tests from four files, one of them run three times by its parameters.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Testing a Data Layer](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/19-testing-a-data-layer.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
